# 数据驱动决策（ACCA 先修课）· Decision Making with Data
## 第 1 周：数据基础与抽样方法

**授课教师**：王奇 副教授（四川大学商学院）  
**研究方向**：人工智能与公司金融  
**邮箱**：qiwangphd@scu.edu.cn

---

### 本周学习目标
1. 理解四种数据测量尺度（名义、定序、间隔、比率）及其区别
2. 掌握六种抽样方法的原理与 Python 模拟
3. 熟练使用 NumPy / Pandas 进行数据创建与分类
4. 建立「数据类型决定统计方法」的核心分析思维

## 0. 导入本节课需要的库

In [ ]:
import numpy as np                                # 导入 NumPy，用于数值计算和数组操作
import pandas as pd                               # 导入 Pandas，用于表格数据处理
import matplotlib.pyplot as plt                   # 导入 Matplotlib 的 pyplot 模块，用于绘图

plt.rcParams['font.sans-serif'] = ['Songti SC', 'SimHei', 'PingFang SC']  # 设置中文字体，避免图表中文乱码
plt.rcParams['axes.unicode_minus'] = False        # 解决负号 '-' 显示为方块的问题

## 1. 数据类型与测量尺度

**核心原则**：数据类型决定适用的统计方法和运算。选错方法，分析结果就没有意义。

In [ ]:
# 用字典模拟一份员工信息表：包含名义、定序、比率等不同尺度的数据
data = {                                          # 创建字典，键为列名，值为列表
    '姓名': ['张三', '李四', '王五', '赵六', '孙七'],                # 姓名：名义数据（仅作标识）
    '部门': ['销售', '技术', '销售', '人事', '技术'],                # 部门：名义数据（类别无顺序）
    '职级': ['初级', '中级', '高级', '初级', '中级'],                # 职级：定序数据（有顺序，间距未知）
    '工龄': [2, 5, 8, 1, 4],                                       # 工龄：比率-离散数据（计数得来）
    '月薪': [8500.0, 12000.0, 18000.0, 6000.0, 11000.0],           # 月薪：比率-连续数据（可精确测量）
    '满意度': ['满意', '非常满意', '一般', '满意', '非常满意']       # 满意度：定序数据（等级有高低）
}

df = pd.DataFrame(data)                           # 用字典创建 Pandas 表格（DataFrame）
df                                                # 在 Notebook 中直接显示表格内容

In [ ]:
df.dtypes                                         # 查看每列自动识别的数据类型（dtypes）

In [ ]:
# 将「部门」列转换为 category（分类）类型，节约内存且语义更准确
df['部门'] = df['部门'].astype('category')         # astype('category') 把文本列转为分类类型

# 将「职级」列转为有序分类：明确指定初级 < 中级 < 高级 的顺序
df['职级'] = pd.Categorical(                       # 用 Categorical 构造有序分类
    df['职级'],                                    # 原始数据列
    categories=['初级', '中级', '高级'],            # 按从低到高列出全部等级
    ordered=True                                  # ordered=True 表示该分类有顺序
)

# 将「满意度」列同样转为有序分类（5 级量表）
df['满意度'] = pd.Categorical(                     # 构造满意度的有序分类
    df['满意度'],                                  # 原始数据列
    categories=['非常不满意', '不满意', '一般', '满意', '非常满意'],  # 从最低到最高的顺序
    ordered=True                                  # 声明为有序分类
)

df.dtypes                                         # 再次查看类型，确认三列已变成 category

In [ ]:
df['职级'].sort_values()                           # 有序分类支持排序：初级排在最前，高级排在最后

### 思考题
- 为什么「名义数据」用数字编码（1=苹果，2=华为）后**不能**做加减运算？
- 一个人的「年龄」是连续型还是离散型？——取决于测量精度！

## 2. NumPy 数组基础

In [ ]:
a = np.array([1, 2, 3, 4, 5])                     # 用列表创建一维数组
print('一维数组:', a)                              # 打印数组内容
print('类型:', type(a))                            # 查看类型：numpy.ndarray

b = np.array([[1, 2, 3], [4, 5, 6]])              # 用嵌套列表创建二维数组（矩阵）
print('二维数组形状:', b.shape)                     # shape 返回 (行数, 列数)，这里是 (2, 3)

In [ ]:
zeros = np.zeros((3, 4))                          # 创建 3 行 4 列的全 0 矩阵
ones = np.ones((2, 2))                            # 创建 2 行 2 列的全 1 矩阵
arange = np.arange(0, 10, 2)                      # 从 0 到 10（不含）步长 2 → [0, 2, 4, 6, 8]
linspace = np.linspace(0, 1, 5)                   # 把 [0, 1] 均分成 5 个数 → [0, 0.25, 0.5, 0.75, 1]

print('zeros:\n', zeros)                          # 打印全 0 矩阵
print('arange:', arange)                          # 打印等差数组
print('linspace:', linspace)                      # 打印均分数组

## 3. 用 Python 模拟六种抽样方法

抽样核心问题：**如何保证样本能够代表总体？**

In [ ]:
np.random.seed(42)                                # 固定随机种子，保证每次运行结果一致（可复现）

# ===== 方法 1：简单随机抽样 =====
population = np.arange(1, 1001)                   # 总体：编号 1~1000 的 1000 人（arange 不含终点）
sample_random = np.random.choice(                 # 从总体中随机抽取
    population,                                   # 抽样的对象：总体数组
    size=50,                                      # 抽取 50 个样本
    replace=False                                 # replace=False 表示不放回抽样（同一人不会被抽中两次）
)
print('简单随机抽样（前10个编号）:', sample_random[:10])   # 打印前 10 个被抽中的编号

In [ ]:
# ===== 方法 2：系统抽样 =====
N, n = 1000, 50                                   # N=总体数量，n=样本数量
k = N // n                                        # 计算抽样间隔 k = 1000 // 50 = 20
r = np.random.randint(1, k + 1)                   # 在 1~20 之间随机选一个起点 r
sample_systematic = np.arange(r, N + 1, k)        # 从起点 r 开始，每隔 k 个取一个：r, r+k, r+2k, ...

print(f'系统抽样：起点 r = {r}，间隔 k = {k}')     # 打印起点和间隔
print('系统抽样（前10个编号）:', sample_systematic[:10])   # 打印被抽中的编号序列

In [ ]:
# ===== 方法 3：分层抽样（按比例分配）=====
strata = {                                        # 用字典定义三个层：层名 → (起始编号, 结束编号)
    '东部': (1, 500),                             # 东部层：编号 1~500，共 500 人
    '中部': (501, 800),                           # 中部层：编号 501~800，共 300 人
    '西部': (801, 1000)                           # 西部层：编号 801~1000，共 200 人
}
sample_sizes = {'东部': 25, '中部': 15, '西部': 10}  # 按人口比例分配样本量：500:300:200 → 25:15:10

sample_stratified = []                            # 用列表收集各层抽出的样本
for name, (start, end) in strata.items():         # 遍历每一层（层名和编号范围）
    layer = np.arange(start, end + 1)             # 生成该层所有成员的编号
    s = np.random.choice(layer, size=sample_sizes[name], replace=False)   # 在层内随机抽样
    sample_stratified.extend(s)                   # 把该层样本追加到总样本列表

sample_stratified = np.array(sample_stratified)   # 转为 NumPy 数组便于后续运算
print('分层抽样总样本量:', len(sample_stratified))  # 应为 25+15+10 = 50

In [ ]:
# ===== 方法 4：整群抽样 =====
schools = np.arange(1, 21)                        # 假设总体分 20 个「群」（如 20 所学校）
chosen_schools = np.random.choice(schools, size=5, replace=False)   # 随机抽中 5 个群（学校）

# 被抽中的群内「全部」成员都调查——这正是整群抽样与分层抽样的本质区别
print('被抽中的学校（群）:', chosen_schools)        # 打印抽中的 5 个学校编号
print('整群抽样：只需去 5 所学校，调查校内全部学生')  # 说明整群抽样的成本优势

In [ ]:
# ===== 方法 5：多阶段抽样（两阶段示意）=====
provinces = ['四川', '广东', '浙江', '江苏']        # 第一阶段单元：省
chosen_province = np.random.choice(provinces, size=1)   # 第一阶段：随机抽 1 个省

cities = [f'{chosen_province[0]}市{i}号' for i in range(1, 11)]   # 第二阶段单元：该省的 10 个市
chosen_city = np.random.choice(cities, size=3, replace=False)     # 第二阶段：再随机抽 3 个市

print('第一阶段抽中的省:', chosen_province[0])      # 打印抽中的省
print('第二阶段抽中的市:', chosen_city)             # 打印进一步抽中的市

In [ ]:
# ===== 方法 6：配额抽样（非概率抽样）=====
# 按总体性别比例设定配额：男 60%、女 40%，共 50 人
quota = {'男': 30, '女': 20}                       # 配额字典：男 30 人、女 20 人

# 调查员按配额「主观选择」符合条件的人——不随机，因此结果不能可靠推断总体
print('配额设置:', quota)                          # 打印配额
print('注意：配额抽样不随机，无法计算抽样误差！')    # 提醒非概率抽样的局限

## 4. 可视化：图表类型必须匹配数据类型

In [ ]:
# ===== 名义数据 → 柱状图（只比多少，不比顺序）=====
categories = ['销售', '技术', '人事', '财务']       # 四个部门（名义类别）
counts = [35, 28, 12, 15]                         # 各部门人数

plt.figure(figsize=(8, 5))                         # 新建画布，宽 8 高 5 英寸
plt.bar(categories, counts,                       # 绘制柱状图：x 为部门，y 为人数
        color=['#C49A6C', '#8B6F4E', '#E8D5C4', '#F5F0E8'])   # 指定四根柱子的颜色
plt.title('各部门人数分布（名义数据 → 柱状图）')     # 设置图表标题
plt.ylabel('人数')                                 # 设置 y 轴标签
plt.show()                                        # 显示图表

In [ ]:
# ===== 连续数值数据 → 直方图（展示分布形状）=====
np.random.seed(42)                                # 固定种子保证可复现
salaries = np.random.normal(10000, 2500, 200)      # 模拟 200 名员工月薪：均值 10000，标准差 2500

plt.figure(figsize=(8, 5))                         # 新建画布
plt.hist(salaries, bins=20,                       # 绘制直方图，把数据分成 20 个区间
         color='#C49A6C', edgecolor='white', alpha=0.85)   # 柱子颜色、白色描边、85% 不透明度
plt.title('员工月薪分布（连续数据 → 直方图）')       # 图表标题
plt.xlabel('月薪（元）')                            # x 轴标签
plt.ylabel('频数')                                 # y 轴标签
plt.show()                                        # 显示图表

In [ ]:
# ===== 定序数据 → 条形图（顺序很重要）=====
levels = ['非常不满意', '不满意', '一般', '满意', '非常满意']   # 5 个满意度等级（从低到高）
level_counts = [5, 12, 25, 48, 30]                # 各等级人数

plt.figure(figsize=(9, 5))                         # 新建画布
colors = ['#8B0000', '#CD5C5C', '#E8D5C4', '#8B6F4E', '#C49A6C']   # 从深红到暖棕的颜色渐变
plt.barh(levels, level_counts, color=colors)      # barh 绘制水平条形图（y 为等级，x 为人数）
plt.title('客户满意度分布（定序数据 → 条形图）')     # 图表标题
plt.xlabel('人数')                                 # x 轴标签
plt.show()                                        # 显示图表

## 5. 本周小结

| 知识点 | 核心内容 |
|---|---|
| 数据类型 | 分类（名义/定序）vs 数值（间隔/比率）|
| 抽样两大类 | 概率抽样可推断总体；非概率抽样快捷但不可靠 |
| 分层 vs 整群 | 每层抽**一部分**（精度）；抽中群内**全部查**（成本）|
| 图表匹配 | 名义→饼图/柱状；定序→条形；连续→直方图 |